
# PPMI RNA-seq Differential Expression Stability Analysis (PyDESeq2 0.4.12)

이 노트북은 **Python 3.9 / PyDESeq2 0.4.12** 환경에 맞춰 작성되었습니다.

주요 반영 사항:
- `DeseqDataSet`는 `design="~condition"` 대신 **`design_factors="condition"`** 방식 사용
- 기준군 설정은 **`ref_level=["condition", "Control"]`** 방식 사용
- `DeseqStats`에는 **`n_cpus`를 넘기지 않음**
- 병렬 설정은 `DefaultInference(n_cpus=...)`에만 전달


In [ ]:

# %pip install -U pydeseq2==0.4.12 scikit-learn pandas numpy matplotlib


In [ ]:

from __future__ import annotations

import re, json, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [ ]:

@dataclass
class AnalysisConfig:
    COUNTS_DIR: str = "./Dataset/RNA/IR3/counts/"
    METADATA_PATH: str = "./Dataset/RNA/IR3/metaDataIR3.csv"
    RESULTS_DIR: str = "./de_stability_results"

    DIAGNOSES: tuple = ("PD", "Control")
    VISITS: tuple = ("BL",)
    QC_PASS_ONLY: bool = True

    ANALYSIS_MODE: str = "kfold" #  "kfold" or "bootstrap"
    N_ITERATIONS_OR_FOLDS: int = 5
    RANDOM_STATE: int = 42

    CONDITION_COL: str = "condition"
    CASE_LABEL: str = "PD"
    CONTROL_LABEL: str = "Control"

    MIN_COUNT_GENE: int = 10
    SIGNIFICANCE_METRIC: str = "padj"
    ALPHA: float = 0.05

    FIT_TYPE: str = "parametric"
    REFIT_COOKS: bool = True
    MIN_REPLICATES: int = 3
    QUIET: bool = True
    LOW_MEMORY: bool = False

    N_CPUS: Optional[int] = None

    COOKS_FILTER: bool = True
    INDEPENDENT_FILTER: bool = True

    KFOLD_SUBSET: str = "train"
    COUNT_FILE_GLOB: str = "*featureCounts*"
    RECURSIVE_SEARCH: bool = False

cfg = AnalysisConfig()
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


In [ ]:

meta_raw = pd.read_csv(cfg.METADATA_PATH)
print("shape:", meta_raw.shape)
print("\nDIAGNOSIS:", sorted(meta_raw["DIAGNOSIS"].dropna().astype(str).unique().tolist()))
print("\nCLINICAL_EVENT:", sorted(meta_raw["CLINICAL_EVENT"].dropna().astype(str).unique().tolist()))
print("\nQCflagIR3:", sorted(meta_raw["QCflagIR3"].dropna().astype(str).unique().tolist()))
display(meta_raw[["PATNO", "DIAGNOSIS", "CLINICAL_EVENT", "QCflagIR3", "HudAlphaID", "Sample"]].head())


In [ ]:

FEATURECOUNTS_SUFFIX_PATTERNS = [
    r"\.featureCounts\.GencodeV29\.txt$",
    r"\.featureCounts\.GencodeV29$",
    r"\.featureCounts-GencodeV29\.txt$",
    r"\.featureCounts-GencodeV29$",
    r"\.featureCounts.*$",
]

def strip_featurecounts_suffix(name: str) -> str:
    out = name
    for pat in FEATURECOUNTS_SUFFIX_PATTERNS:
        out = re.sub(pat, "", out)
    return out

def normalize_sample_key(x: object) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = Path(s).name
    s = re.sub(r"\.txt$", "", s)
    s = strip_featurecounts_suffix(s)
    return s

def parse_count_filename(path: str | Path) -> Dict[str, str]:
    name = Path(path).name
    sample_key = normalize_sample_key(name)
    parts = sample_key.split(".")
    return {
        "file_name": name,
        "sample_key": sample_key,
        "study_release": parts[0] if len(parts) > 0 else "",
        "PATNO_file": parts[1] if len(parts) > 1 else "",
        "CLINICAL_EVENT_file": parts[2] if len(parts) > 2 else "",
        "SampleID_file": parts[3] if len(parts) > 3 else "",
        "HudAlphaID_file": parts[4] if len(parts) > 4 else "",
    }

def read_featurecounts_file(path: str | Path) -> pd.Series:
    path = Path(path)
    df = pd.read_csv(path, sep="\t", comment="#", low_memory=False)
    if df.empty:
        raise ValueError(f"Empty featureCounts file: {path}")
    possible_gene_cols = ["Geneid", "gene_id", "GeneID", "gene"]
    gene_col = next((c for c in possible_gene_cols if c in df.columns), df.columns[0])
    count_col = df.columns[-1]
    counts = pd.Series(df[count_col].values, index=df[gene_col].astype(str).values, name=normalize_sample_key(path.name))
    counts = pd.to_numeric(counts, errors="coerce").fillna(0).astype(int)
    return counts

def build_counts_matrix(counts_dir: str | Path, file_glob: str = "*featureCounts*", recursive: bool = False):
    counts_dir = Path(counts_dir)
    files = sorted(counts_dir.rglob(file_glob) if recursive else counts_dir.glob(file_glob))
    if not files:
        raise FileNotFoundError(
            f"No files matched {file_glob!r} under {counts_dir}. Check cfg.COUNTS_DIR, cfg.COUNT_FILE_GLOB, and cfg.RECURSIVE_SEARCH."
        )
    sample_series, file_records = [], []
    for fp in files:
        sample_series.append(read_featurecounts_file(fp))
        file_records.append(parse_count_filename(fp))
    counts_df = pd.DataFrame(sample_series)
    counts_df.index = [s.name for s in sample_series]
    counts_df.index.name = "sample_key"
    counts_df.columns.name = "gene"
    file_info = pd.DataFrame(file_records)
    return counts_df, file_info

def prepare_metadata(meta_df: pd.DataFrame, diagnoses: Iterable[str], visits: Iterable[str], qc_pass_only: bool = True) -> pd.DataFrame:
    meta = meta_df.copy()
    meta["PATNO"] = meta["PATNO"].astype(str)
    meta["DIAGNOSIS"] = meta["DIAGNOSIS"].astype(str)
    meta["CLINICAL_EVENT"] = meta["CLINICAL_EVENT"].astype(str)
    meta["QCflagIR3"] = meta["QCflagIR3"].astype(str)
    meta["HudAlphaID"] = meta["HudAlphaID"].astype(str)
    meta["normalized_sample_key"] = meta["Sample"].map(normalize_sample_key)
    meta = meta[meta["DIAGNOSIS"].isin(list(diagnoses))]
    meta = meta[meta["CLINICAL_EVENT"].isin(list(visits))]
    if qc_pass_only:
        meta = meta[meta["QCflagIR3"].str.lower().eq("pass")]
    return meta.reset_index(drop=True)

def match_counts_to_metadata(counts_df: pd.DataFrame, file_info: pd.DataFrame, meta_df: pd.DataFrame):
    meta = meta_df.copy()
    files = file_info.copy()
    files["sample_key"] = files["sample_key"].astype(str)
    files["HudAlphaID_file"] = files["HudAlphaID_file"].astype(str)
    meta["normalized_sample_key"] = meta["normalized_sample_key"].astype(str)
    meta["HudAlphaID"] = meta["HudAlphaID"].astype(str)
    files = files.set_index("sample_key").loc[counts_df.index].reset_index()
    exact = files.merge(meta, left_on="sample_key", right_on="normalized_sample_key", how="left", suffixes=("", "_meta"))
    need_fallback = exact["PATNO"].isna()
    if need_fallback.any():
        fallback = files.loc[need_fallback, ["sample_key", "HudAlphaID_file"]].merge(meta, left_on="HudAlphaID_file", right_on="HudAlphaID", how="left", suffixes=("", "_meta"))
        fallback = fallback.set_index("sample_key")
        exact = exact.set_index("sample_key")
        fill_cols = [c for c in fallback.columns if c in exact.columns]
        for col in fill_cols:
            exact.loc[fallback.index, col] = exact.loc[fallback.index, col].combine_first(fallback[col])
        exact = exact.reset_index()
    matched = exact[~exact["PATNO"].isna()].copy()
    if matched.empty:
        raise ValueError("No count files were matched to metadata.")
    counts_matched = counts_df.loc[matched["sample_key"]].copy()
    metadata_matched = matched.copy()
    metadata_matched["analysis_sample_id"] = metadata_matched["sample_key"].astype(str)
    metadata_matched = metadata_matched.set_index("analysis_sample_id")
    counts_matched.index = metadata_matched.index
    metadata_matched["condition"] = metadata_matched["DIAGNOSIS"].astype(str)
    return counts_matched, metadata_matched, matched

def load_and_prepare_data(cfg: AnalysisConfig):
    counts_df, file_info = build_counts_matrix(cfg.COUNTS_DIR, cfg.COUNT_FILE_GLOB, recursive=cfg.RECURSIVE_SEARCH)
    meta_raw = pd.read_csv(cfg.METADATA_PATH)
    meta_filtered = prepare_metadata(meta_raw, diagnoses=cfg.DIAGNOSES, visits=cfg.VISITS, qc_pass_only=cfg.QC_PASS_ONLY)
    counts_matched, metadata_matched, matched_table = match_counts_to_metadata(counts_df, file_info, meta_filtered)
    valid = metadata_matched["condition"].isin([cfg.CASE_LABEL, cfg.CONTROL_LABEL])
    counts_matched = counts_matched.loc[valid]
    metadata_matched = metadata_matched.loc[valid]
    return counts_matched, metadata_matched, matched_table


In [ ]:

def _make_unique_index(ids: List[str], prefix: str) -> List[str]:
    return [f"{prefix}__{i:04d}__{sid}" for i, sid in enumerate(ids)]

def make_bootstrap_subsets(counts_df, meta_df, condition_col, n_iterations, random_state=42):
    rng = np.random.default_rng(random_state)
    groups = meta_df.groupby(condition_col)
    for i in range(1, n_iterations + 1):
        sampled_ids = []
        for _, sub in groups:
            ids = sub.index.to_numpy()
            sampled_ids.extend(rng.choice(ids, size=len(ids), replace=True).tolist())
        sub_counts = counts_df.loc[sampled_ids].copy()
        sub_meta = meta_df.loc[sampled_ids].copy()
        new_index = _make_unique_index(sampled_ids, prefix=f"bootstrap_{i}")
        sub_counts.index = new_index
        sub_meta.index = new_index
        yield {"iteration_id": f"bootstrap_{i}", "subset_type": "bootstrap", "counts": sub_counts, "metadata": sub_meta}

def make_kfold_subsets(counts_df, meta_df, condition_col, n_splits, random_state=42, subset="train"):
    y = meta_df[condition_col].astype(str).values
    sample_ids = meta_df.index.to_numpy()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(skf.split(sample_ids, y), start=1):
        idx = train_idx if subset == "train" else test_idx
        selected_ids = sample_ids[idx].tolist()
        yield {"iteration_id": f"kfold_{subset}_{fold}", "subset_type": f"kfold_{subset}", "counts": counts_df.loc[selected_ids].copy(), "metadata": meta_df.loc[selected_ids].copy()}

def generate_subsets(counts_df, meta_df, cfg):
    if cfg.ANALYSIS_MODE == "bootstrap":
        yield from make_bootstrap_subsets(counts_df, meta_df, cfg.CONDITION_COL, cfg.N_ITERATIONS_OR_FOLDS, cfg.RANDOM_STATE)
    elif cfg.ANALYSIS_MODE == "kfold":
        yield from make_kfold_subsets(counts_df, meta_df, cfg.CONDITION_COL, cfg.N_ITERATIONS_OR_FOLDS, cfg.RANDOM_STATE, cfg.KFOLD_SUBSET)
    else:
        raise ValueError("cfg.ANALYSIS_MODE must be 'bootstrap' or 'kfold'")


In [ ]:

def filter_genes_by_min_count(counts_df: pd.DataFrame, min_count_gene: int) -> pd.DataFrame:
    keep = counts_df.sum(axis=0) >= int(min_count_gene)
    return counts_df.loc[:, keep]

def validate_subset_for_de(meta_df: pd.DataFrame, condition_col: str, case_label: str, control_label: str) -> None:
    vc = meta_df[condition_col].astype(str).value_counts()
    if case_label not in vc.index or control_label not in vc.index:
        raise ValueError(f"Both labels must be present. Got counts: {vc.to_dict()}")
    if (vc[[case_label, control_label]] < 2).any():
        raise ValueError(f"Each group should have at least 2 samples. Got counts: {vc.to_dict()}")

def run_single_deseq_v0412(counts_df: pd.DataFrame, meta_df: pd.DataFrame, cfg: AnalysisConfig):
    validate_subset_for_de(meta_df, cfg.CONDITION_COL, cfg.CASE_LABEL, cfg.CONTROL_LABEL)
    counts_use = filter_genes_by_min_count(counts_df, cfg.MIN_COUNT_GENE).copy().astype(int)
    meta_use = meta_df.copy()
    meta_use[cfg.CONDITION_COL] = meta_use[cfg.CONDITION_COL].astype(str)
    inference = DefaultInference(n_cpus=cfg.N_CPUS)
    dds = DeseqDataSet(
        counts=counts_use,
        metadata=meta_use,
        design_factors=cfg.CONDITION_COL,
        ref_level=[cfg.CONDITION_COL, cfg.CONTROL_LABEL],
        fit_type=cfg.FIT_TYPE,
        refit_cooks=cfg.REFIT_COOKS,
        min_replicates=cfg.MIN_REPLICATES,
        n_cpus=cfg.N_CPUS,
        inference=inference,
        quiet=cfg.QUIET,
        low_memory=cfg.LOW_MEMORY,
    )
    dds.deseq2()
    stat_res = DeseqStats(
        dds,
        contrast=[cfg.CONDITION_COL, cfg.CASE_LABEL, cfg.CONTROL_LABEL],
        alpha=cfg.ALPHA,
        cooks_filter=cfg.COOKS_FILTER,
        independent_filter=cfg.INDEPENDENT_FILTER,
        inference=inference,
        quiet=cfg.QUIET,
    )
    stat_res.summary()
    res = stat_res.results_df.copy()
    res.index.name = "gene"
    res = res.reset_index()
    metric = cfg.SIGNIFICANCE_METRIC
    if metric not in {"pvalue", "padj"}:
        raise ValueError("SIGNIFICANCE_METRIC must be 'pvalue' or 'padj'")
    res["is_significant"] = res[metric].notna() & (res[metric] < cfg.ALPHA)
    return {"dds": dds, "stat_res": stat_res, "results_df": res, "n_samples": meta_use.shape[0], "n_genes_input": counts_df.shape[1], "n_genes_tested": counts_use.shape[1], "group_counts": meta_use[cfg.CONDITION_COL].value_counts().to_dict()}


In [ ]:

def run_repeated_de_analysis_v0412(counts_df, meta_df, cfg):
    iteration_summaries, all_results, errors = [], [], []
    for subset in generate_subsets(counts_df, meta_df, cfg):
        iteration_id = subset["iteration_id"]
        try:
            out = run_single_deseq_v0412(subset["counts"], subset["metadata"], cfg)
            res = out["results_df"].copy()
            res["iteration_id"] = iteration_id
            res["subset_type"] = subset["subset_type"]
            res["n_samples"] = out["n_samples"]
            res["n_genes_input"] = out["n_genes_input"]
            res["n_genes_tested"] = out["n_genes_tested"]
            group_counts = out["group_counts"]
            res["n_case"] = group_counts.get(cfg.CASE_LABEL, 0)
            res["n_control"] = group_counts.get(cfg.CONTROL_LABEL, 0)
            all_results.append(res)
            iteration_summaries.append({
                "iteration_id": iteration_id,
                "subset_type": subset["subset_type"],
                "status": "ok",
                "n_samples": out["n_samples"],
                "n_genes_input": out["n_genes_input"],
                "n_genes_tested": out["n_genes_tested"],
                "n_case": group_counts.get(cfg.CASE_LABEL, 0),
                "n_control": group_counts.get(cfg.CONTROL_LABEL, 0),
                "n_significant": int(res["is_significant"].sum()),
                "median_abs_log2fc": float(res["log2FoldChange"].abs().median(skipna=True)),
            })
        except Exception as e:
            errors.append({"iteration_id": iteration_id, "error": str(e)})
            iteration_summaries.append({"iteration_id": iteration_id, "subset_type": subset["subset_type"], "status": "error", "error": str(e)})
    iteration_summary_df = pd.DataFrame(iteration_summaries)
    all_results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    errors_df = pd.DataFrame(errors)
    return iteration_summary_df, all_results_df, errors_df

def summarize_gene_stability(all_results_df: pd.DataFrame) -> pd.DataFrame:
    if all_results_df.empty:
        return pd.DataFrame()
    grouped = all_results_df.groupby("gene", dropna=False)
    summary = grouped.agg(
        runs_present=("iteration_id", "nunique"),
        sig_frequency=("is_significant", "mean"),
        mean_log2fc=("log2FoldChange", "mean"),
        std_log2fc=("log2FoldChange", "std"),
        median_log2fc=("log2FoldChange", "median"),
        mean_pvalue=("pvalue", "mean"),
        median_pvalue=("pvalue", "median"),
        mean_padj=("padj", "mean"),
        median_padj=("padj", "median"),
        mean_baseMean=("baseMean", "mean"),
        median_baseMean=("baseMean", "median"),
    ).reset_index()
    summary["abs_mean_log2fc"] = summary["mean_log2fc"].abs()
    return summary.sort_values(["sig_frequency", "abs_mean_log2fc", "median_baseMean"], ascending=[False, False, False]).reset_index(drop=True)

def compute_jaccard_matrix(all_results_df: pd.DataFrame) -> pd.DataFrame:
    if all_results_df.empty:
        return pd.DataFrame()
    sig_sets = {it: set(df.loc[df["is_significant"], "gene"].astype(str)) for it, df in all_results_df.groupby("iteration_id")}
    iterations = list(sig_sets.keys())
    mat = pd.DataFrame(index=iterations, columns=iterations, dtype=float)
    for i in iterations:
        for j in iterations:
            a, b = sig_sets[i], sig_sets[j]
            denom = len(a | b)
            mat.loc[i, j] = np.nan if denom == 0 else len(a & b) / denom
    return mat


In [ ]:

def plot_iteration_summary(iteration_summary_df: pd.DataFrame):
    ok_df = iteration_summary_df[iteration_summary_df["status"] == "ok"].copy()
    if ok_df.empty:
        print("No successful iterations to plot.")
        return
    plt.figure(figsize=(10, 5))
    plt.bar(ok_df["iteration_id"], ok_df["n_significant"])
    plt.xticks(rotation=90)
    plt.ylabel("Number of significant genes")
    plt.title("Significant gene counts per iteration")
    plt.tight_layout(); plt.show()
    plt.figure(figsize=(10, 5))
    plt.bar(ok_df["iteration_id"], ok_df["median_abs_log2fc"])
    plt.xticks(rotation=90)
    plt.ylabel("Median |log2FC|")
    plt.title("Median absolute log2 fold change per iteration")
    plt.tight_layout(); plt.show()

def plot_top_stable_genes(gene_stability_df: pd.DataFrame, top_n: int = 20):
    if gene_stability_df.empty:
        print("Empty gene stability table.")
        return
    top = gene_stability_df.head(top_n).iloc[::-1]
    plt.figure(figsize=(8, max(6, top_n * 0.35)))
    plt.barh(top["gene"], top["sig_frequency"])
    plt.xlabel("Significant frequency across iterations")
    plt.ylabel("Gene")
    plt.title(f"Top {top_n} stable genes")
    plt.tight_layout(); plt.show()

def plot_jaccard_heatmap(jaccard_df: pd.DataFrame):
    if jaccard_df.empty:
        print("Empty Jaccard matrix.")
        return
    plt.figure(figsize=(8, 6))
    plt.imshow(jaccard_df.values, aspect="auto")
    plt.colorbar(label="Jaccard similarity")
    plt.xticks(range(len(jaccard_df.columns)), jaccard_df.columns, rotation=90)
    plt.yticks(range(len(jaccard_df.index)), jaccard_df.index)
    plt.title("Jaccard similarity of significant gene sets")
    plt.tight_layout(); plt.show()


In [ ]:

cfg = AnalysisConfig(
    # COUNTS_DIR="./counts",
    # METADATA_PATH="./metaDataIR3.csv",
    # RESULTS_DIR="./de_stability_results_v0412",
    # DIAGNOSES=("PD", "Control"),
    # VISITS=("BL",),
    # QC_PASS_ONLY=True,
    # ANALYSIS_MODE="kfold",
    # N_ITERATIONS_OR_FOLDS=5,
    # RANDOM_STATE=42,
    # CONDITION_COL="condition",
    # CASE_LABEL="PD",
    # CONTROL_LABEL="Control",
    # MIN_COUNT_GENE=10,
    # SIGNIFICANCE_METRIC="padj",
    # ALPHA=0.05,
    # FIT_TYPE="parametric",
    # REFIT_COOKS=True,
    # MIN_REPLICATES=3,
    # QUIET=True,
    # LOW_MEMORY=False,
    # N_CPUS=4,
    # COOKS_FILTER=True,
    # INDEPENDENT_FILTER=True,
    # KFOLD_SUBSET="train",
    # COUNT_FILE_GLOB="*featureCounts*",
    # RECURSIVE_SEARCH=False,
)
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


In [ ]:

counts_all, meta_all, matched_table = load_and_prepare_data(cfg)
print("counts_all shape:", counts_all.shape)
print("meta_all shape:", meta_all.shape)
print("\nGroup counts:")
print(meta_all["condition"].value_counts(dropna=False))
display(matched_table.head())
display(meta_all[["condition", "CLINICAL_EVENT", "QCflagIR3", "PATNO", "HudAlphaID"]].head())


In [ ]:
for subset in generate_subsets(counts_all, meta_all, cfg):
    print(len(subset["counts"]), len(subset["metadata"]), subset["iteration_id"], subset["subset_type"])

In [ ]:

iteration_summary_df, all_results_df, errors_df = run_repeated_de_analysis_v0412(counts_all, meta_all, cfg)
print("iteration_summary_df shape:", iteration_summary_df.shape)
print("all_results_df shape:", all_results_df.shape)
print("errors_df shape:", errors_df.shape)
display(iteration_summary_df)
if not errors_df.empty:
    display(errors_df)


In [ ]:

gene_stability_df = summarize_gene_stability(all_results_df)
jaccard_df = compute_jaccard_matrix(all_results_df)
display(gene_stability_df.head(30))
display(jaccard_df)


In [ ]:

results_dir = Path(cfg.RESULTS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)
matched_table.to_csv(results_dir / "matched_samples.csv", index=False)
iteration_summary_df.to_csv(results_dir / "iteration_summary.csv", index=False)
all_results_df.to_csv(results_dir / "all_gene_results.csv.gz", index=False, compression="gzip")
gene_stability_df.to_csv(results_dir / "gene_stability_summary.csv", index=False)
jaccard_df.to_csv(results_dir / "jaccard_similarity.csv")
with open(results_dir / "analysis_config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)
print(f"Saved outputs to: {results_dir.resolve()}")


In [ ]:

plot_iteration_summary(iteration_summary_df)
plot_top_stable_genes(gene_stability_df, top_n=20)
plot_jaccard_heatmap(jaccard_df)
